- ## _Data Loading_

**_Set up Azure Storage access using SAS key:_**

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
spark.conf.set("fs.azure.account.auth.type.handsonproject1.dfs.core.windows.net", "SAS")
spark.conf.set("fs.azure.sas.token.provider.type.handsonproject1.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.sas.FixedSASTokenProvider")
spark.conf.set("fs.azure.sas.fixed.token.handsonproject1.dfs.core.windows.net", "sv=2026-02-06&ss=bfqt&srt=sco&sp=rwdlacupyx&se=2026-09-03T06:50:34Z&st=2026-08-02T22:35:34Z&spr=https&sig=gkJBcq4M3898kQ98jUcWKFqdlZLMCdc92Pv4PCYVWTI%3D")

_**Read raw data from storage into DataFrame:**_

In [0]:
df_customers= spark.read.format("csv")\
    .option("header", "true")\
    .option("inferSchema", "True")\
    .load("abfss://bronze@handsonproject1.dfs.core.windows.net/customers/customers.csv")
display(df_customers)

In [0]:
df_products= spark.read.format("csv")\
    .option("header", "true")\
    .option("inferSchema", "True")\
    .load("abfss://bronze@handsonproject1.dfs.core.windows.net/products/products.csv")
display(df_products)

In [0]:
df_prod_translation= spark.read.format("csv")\
    .option("header", "true")\
    .option("inferSchema", "True")\
    .load("abfss://bronze@handsonproject1.dfs.core.windows.net/product_category_name_translation/product_category_name_translation.csv")
display(df_prod_translation)

In [0]:
df_order_items= spark.read.format("csv")\
    .option("header", "true")\
    .option("inferSchema", "True")\
    .load("abfss://bronze@handsonproject1.dfs.core.windows.net/order_items/order_items.csv")
display(df_order_items)

In [0]:
df_geolocation= spark.read.format("csv")\
    .option("header", "true")\
    .option("inferSchema", "True")\
    .load("abfss://bronze@handsonproject1.dfs.core.windows.net/geolocation/geolocation.csv")
display(df_geolocation)

In [0]:
df_order_payments= spark.read.format("csv")\
    .option("header", "true")\
    .option("inferSchema", "True")\
    .load("abfss://bronze@handsonproject1.dfs.core.windows.net/order_payments/order_payments.csv")
display(df_order_payments)

In [0]:
df_order_reviews= spark.read.format("csv")\
    .option("header", "true")\
    .option("inferSchema", "True")\
    .option("multiLine", "true")\
    .option("quote", '"')\
    .option("escape", '"') \
    .load("abfss://bronze@handsonproject1.dfs.core.windows.net/order_reviews/order_reviews.csv")
display(df_order_reviews)

In [0]:
df_orders= spark.read.format("csv")\
    .option("header", "true")\
    .option("inferSchema", "True")\
    .load("abfss://bronze@handsonproject1.dfs.core.windows.net/orders/orders.csv")
display(df_orders)

In [0]:
df_sellers= spark.read.format("csv")\
    .option("header", "true")\
    .option("inferSchema", "True")\
    .load("abfss://bronze@handsonproject1.dfs.core.windows.net/sellers/sellers.csv")
display(df_sellers)

- ## _Data Transformation_

_GEOLOCATION TABLE:_

In [0]:
df_geolocation.printSchema()

In [0]:
df_geolocation= df_geolocation.withColumn(
    "geolocation_zip_code_prefix", 
    lpad(col("geolocation_zip_code_prefix").cast("string"), 5, "0"))

In [0]:
df_geolocation.filter(length(col("geolocation_zip_code_prefix")) < 5).display()

In [0]:
df_geolocation= df_geolocation.withColumn("geolocation_city", lower(trim(col("geolocation_city"))))\
                              .withColumn("geolocation_state", upper(trim(col("geolocation_state"))))   

In [0]:
# Write the cleaned and updated sellers DataFrame to the Silver Layer
df_geolocation.coalesce(1).write.format("delta")\
    .mode("overwrite")\
    .option("path","abfss://silver@handsonproject1.dfs.core.windows.net/geolocation")\
    .save()

_CUSTOMERS TABLE:_

In [0]:
df_customers.printSchema()

In [0]:
df_customers.toPandas().isna().sum()

In [0]:
id_columns = ["customer_id", "customer_unique_id"]
for c in id_columns:
    df_customers = df_customers.withColumn(c, trim(col(c)))

df_customers= df_customers.withColumn("customer_city", lower(trim(col("customer_city"))))\
                          .withColumn("customer_state", upper(trim(col("customer_state"))))   

In [0]:
df_customers= df_customers.withColumn(
    "customer_zip_code_prefix", 
    lpad(col("customer_zip_code_prefix").cast("string"), 5, "0"))

display(df_customers) 

In [0]:
df_customers.filter(length(col("customer_zip_code_prefix")) < 5).display()

In [0]:
# Write the cleaned and updated customers DataFrame to the Silver Layer
df_customers.coalesce(1).write.format("delta")\
    .mode("overwrite")\
    .option("path","abfss://silver@handsonproject1.dfs.core.windows.net/customers")\
    .save()

In [0]:
display(spark.read.format("delta").load("abfss://silver@handsonproject1.dfs.core.windows.net/customers"))

_PRODUCT_CATEGORY_NAME_TRANSLATION TABLE:_

In [0]:
df_prod_translation.printSchema()

In [0]:
text_columns = ["product_category_name", "product_category_name_english"]
for c in text_columns:
    df_prod_translation = df_prod_translation.withColumn(c, lower(trim(col(c))))

In [0]:
df_prod_translation.toPandas().isna().sum()

In [0]:
# Write the cleaned and updated order_reviews DataFrame to the Silver Layer
df_prod_translation.coalesce(1).write.format("delta")\
    .mode("overwrite")\
    .option("path","abfss://silver@handsonproject1.dfs.core.windows.net/product_category_name_translation")\
    .save()

_PRODUCTS TABLE:_

In [0]:
df_products.printSchema()

In [0]:
df_products= df_products.withColumn("product_category_name", lower(trim(col("product_category_name"))))
df_products= df_products.withColumn("product_id", lower(col("product_id")))

In [0]:
df_products.toPandas().isna().sum()

In [0]:
#Check if these 610 missing products were actually sold or just input errors

df_order_items\
    .join(df_products, on="product_id")\
    .join(df_orders, on="order_id")\
    .filter(col("product_category_name").isNull())\
    .select(countDistinct("product_id"))\
    .display()


In [0]:
df_order_items\
    .join(df_products, on="product_id")\
    .join(df_orders, on="order_id")\
    .filter(col("product_category_name").isNull())\
    .groupBy("order_status")\
    .agg(count("order_status"))\
    .display()

In [0]:
#610 unique products generated over 1,500 'delivered' orders so we will Keep missing products and mark as 'unknown'
df_products = df_products \
    .withColumn("product_category_name", 
                when(col("product_category_name").isNull(), lit("unknown"))
                .otherwise(col("product_category_name"))) 

display(df_products)

In [0]:
#View details to verify if the missing values belong to the exact same 2 products 
df_products.filter(
    col("product_length_cm").isNull() |
    col("product_width_cm").isNull() |
    col("product_height_cm").isNull() |
    col("product_weight_g").isNull())\
    .display()

In [0]:
df_products.filter(
    (col("product_weight_g") <= 0) |
    (col("product_length_cm") <= 0) |
    (col("product_height_cm") <= 0) |
    (col("product_width_cm") <= 0) |
    (col("product_name_lenght") <= 0) |
    (col("product_description_lenght") <= 0) |
    (col("product_photos_qty") <= 0))\
    .display()

In [0]:
#Convert invalid 0 weights to NULL
df_products = df_products.withColumn(
    "product_weight_g", when(col("product_weight_g") <= 0, None)\
    .otherwise(col("product_weight_g")))

In [0]:
# Write the cleaned and updated products DataFrame to the Silver Layer
df_products.coalesce(1).write.format("delta")\
    .mode("overwrite")\
    .option("path","abfss://silver@handsonproject1.dfs.core.windows.net/products")\
    .save()

_SELLERS TABLE:_

In [0]:
df_sellers.printSchema()

In [0]:
df_sellers.toPandas().isna().sum()

In [0]:
df_sellers= df_sellers.withColumn("seller_id", trim(col("seller_id")))\
                      .withColumn("seller_city", lower(trim(col("seller_city"))))\
                      .withColumn("seller_state", upper(trim(col("seller_state"))))   

In [0]:
df_sellers= df_sellers.withColumn(
    "seller_zip_code_prefix", 
    lpad(col("seller_zip_code_prefix").cast("string"), 5, "0"))

In [0]:
df_sellers.filter(length(col("seller_zip_code_prefix")) < 5).display()

In [0]:
# Write the cleaned and updated sellers DataFrame to the Silver Layer
df_sellers.coalesce(1).write.format("delta")\
    .mode("overwrite")\
    .option("path","abfss://silver@handsonproject1.dfs.core.windows.net/sellers")\
    .save()

_ORDERS TABLE:_

In [0]:
df_orders.printSchema()

In [0]:
id_columns = ["order_id","customer_id"]
for c in id_columns:
    df_orders = df_orders.withColumn(c, trim(col(c)))

df_orders= df_orders.withColumn("order_status", lower(trim(col("order_status"))))   

In [0]:
df_orders.toPandas().isna().sum()

In [0]:
df_orders.groupBy("order_status").agg(
    sum(when(col("order_delivered_customer_date").isNull(), 1).otherwise(0)).alias("missing_delivery_date"),
    count("*").alias("total_orders_in_status")).display()

In [0]:
#Check the 8 'delivered' orders that lack a customer delivery date
df_orders.filter(
    (col("order_status") == "delivered") & 
    (col("order_delivered_customer_date").isNull()))\
    .display()

In [0]:
#Check the 6 'canceled' orders with a delivery date.
df_orders.filter(
    (col("order_status") == "canceled") & 
    (col("order_delivered_customer_date").isNotNull()))\
    .display()


#Note: Left as is because these are very few rows representing system errors or returns, 
        #so they won't impact the final analysis

In [0]:
# Write the cleaned and updated order DataFrame to the Silver Layer
df_orders.coalesce(1).write.format("delta")\
    .mode("overwrite")\
    .option("path","abfss://silver@handsonproject1.dfs.core.windows.net/orders")\
    .save()

_ORDER_ITEMS TABLE:_

In [0]:
df_order_items.printSchema()

In [0]:
id_columns = ["order_id","product_id","seller_id"]
for c in id_columns:
    df_order_items = df_order_items.withColumn(c, trim(col(c)))

In [0]:
df_order_items.toPandas().isna().sum()

In [0]:
df_order_items.filter(
    (col("price") <= 0)|
    (col("freight_value") <= 0)).display()

# Note: 383 items appear to have a freight_value of 0, likely indicating Free Shipping

In [0]:
# Write the cleaned and updated order_items DataFrame to the Silver Layer
df_order_items.coalesce(1).write.format("delta")\
    .mode("overwrite")\
    .option("path","abfss://silver@handsonproject1.dfs.core.windows.net/order_items")\
    .save()

_ORDER PAYMEMTS TABLE:_

In [0]:
df_order_payments.printSchema()

In [0]:
df_order_payments = df_order_payments.withColumn("order_id", trim(col("order_id")))\
                                     .withColumn("payment_type", lower(trim(col("payment_type"))))

In [0]:
df_order_payments.toPandas().isna().sum()

In [0]:
df_order_payments.select("payment_type").distinct().display()

In [0]:
#Verifying if 'not_defined' payments have matching records in orders and items tables
df_order_items.alias("oi")\
    .join(df_orders.alias("o"), col("oi.order_id") == col("o.order_id"), how="right")\
    .join(df_order_payments.alias("op"), col("oi.order_id") == col("op.order_id"), how="right")\
    .filter(col("payment_type") == "not_defined")\
    .select(col("oi.order_id").alias("order_id_orders_table"),
             "op.order_id",
             "o.order_status",
             "op.payment_type",
             "op.payment_value")\
    .display()

In [0]:
df_order_payments= df_order_payments.filter(
    col("payment_type") != "not_defined")

In [0]:
df_order_payments.filter(col("payment_value") <= 0).display()


# Note: Verified that these 0 values are extra vouchers used in the same order
       #total amount paid is correct with no financial impact

In [0]:
# Write the cleaned and updated order_payments DataFrame to the Silver Layer
df_order_payments.coalesce(1).write.format("delta")\
    .mode("overwrite")\
    .option("path","abfss://silver@handsonproject1.dfs.core.windows.net/order_payments")\
    .save()

_ORDER REVIEWS TABLE:_

In [0]:
df_order_reviews.printSchema()

In [0]:
trim_columns = ["review_id","order_id","review_comment_title","review_comment_message"]
for c in trim_columns:
    df_order_reviews = df_order_reviews.withColumn(c, trim(col(c)))

In [0]:
df_order_reviews.toPandas().isna().sum()

In [0]:
# Write the cleaned and updated order_reviews DataFrame to the Silver Layer
df_order_reviews.coalesce(1).write.format("delta")\
    .mode("overwrite")\
    .option("path","abfss://silver@handsonproject1.dfs.core.windows.net/order_reviews")\
    .save()